# Agentic RAG with citation-preserving guest retrieval

An original notebook adaptation of the course's Space-based gala project. The required path is deterministic; live data and web enrichment are visibly separate.

## Learning objectives

- Build a stable synthetic guest corpus with durable citation IDs.
- Implement tokenization, BM25-style ranking, a retriever-tool boundary, and grounded composition.
- Evaluate top-one retrieval and citation preservation without a model.
- Keep optional web enrichment distinct from retrieved guest facts.

## Setup

The offline path uses only the Python standard library and never opens the network. For the optional cells, install packages with `python -m pip install datasets smolagents huggingface_hub requests` and, for toolkit search, `python -m pip install 'smolagents[toolkit]'`. Live cells require explicit `ENABLE_LIVE_NETWORK=1` plus the credentials named in each guard.

## Complete implementation

Run the next cell to define the complete deterministic project pipeline.

In [1]:
import math
import re
from collections import Counter

# The records are intentionally synthetic, small, and stable. Their IDs are the
# citation keys; optional web material is never added to this corpus.
guest_corpus = [
    {
        "id": "guest-ada",
        "name": "Ada Lovelace",
        "relation": "mathematics guest",
        "description": "Worked on Charles Babbage's Analytical Engine and is associated with early computer programming.",
        "conversation_topics": ["mathematics", "early computing"],
        "email": "ada@example.invalid",
    },
    {
        "id": "guest-grace",
        "name": "Grace Hopper",
        "relation": "computing guest",
        "description": "Programming pioneer known for compiler work and clear explanations of computer systems.",
        "conversation_topics": ["compilers", "naval computing"],
        "email": "grace@example.invalid",
    },
    {
        "id": "guest-katherine",
        "name": "Katherine Johnson",
        "relation": "aerospace guest",
        "description": "Mathematician whose trajectory calculations supported early crewed spaceflight.",
        "conversation_topics": ["orbital mechanics", "spaceflight"],
        "email": "katherine@example.invalid",
    },
]

def tokenize(text: str) -> list[str]:
    """Return stable lowercase word tokens for the lexical retriever."""
    return re.findall(r"[a-z0-9]+", text.casefold())

def record_text(record: dict) -> str:
    return " ".join(
        [
            record["name"],
            record["relation"],
            record["description"],
            *record["conversation_topics"],
        ]
    )

document_tokens = [tokenize(record_text(record)) for record in guest_corpus]
document_frequencies = Counter(
    token for tokens in document_tokens for token in set(tokens)
)
average_document_length = sum(map(len, document_tokens)) / len(document_tokens)

def bm25_score(query: str, document_index: int, *, k1: float = 1.5, b: float = 0.75) -> float:
    """Compute a compact BM25-style lexical relevance score."""
    query_tokens = tokenize(query)
    tokens = document_tokens[document_index]
    term_frequencies = Counter(tokens)
    score = 0.0
    for term in query_tokens:
        frequency = term_frequencies[term]
        if not frequency:
            continue
        document_frequency = document_frequencies[term]
        inverse_document_frequency = math.log(
            1 + (len(guest_corpus) - document_frequency + 0.5) / (document_frequency + 0.5)
        )
        denominator = frequency + k1 * (
            1 - b + b * len(tokens) / average_document_length
        )
        score += inverse_document_frequency * frequency * (k1 + 1) / denominator
    return score

def retrieve_guests(query: str, limit: int = 3) -> list[dict]:
    """Return scored copies of records in deterministic rank order."""
    if limit < 1:
        raise ValueError("limit must be at least one")
    scored = []
    for index, record in enumerate(guest_corpus):
        result = dict(record)
        result["score"] = round(bm25_score(query, index), 8)
        scored.append(result)
    return sorted(scored, key=lambda item: (-item["score"], item["id"]))[:limit]

def guest_info_retriever(query: str) -> list[dict]:
    """Tool boundary exposed to an agent: retrieve, but do not invent, records."""
    return retrieve_guests(query)

def compose_grounded_answer(query: str, live_enrichment: list[dict] | None = None) -> dict:
    """Compose from a retrieved record and preserve its stable citation ID."""
    top_record = guest_info_retriever(query)[0]
    if top_record["score"] <= 0:
        return {
            "answer": "No matching guest record was found.",
            "sources": [],
            "live_enrichment": list(live_enrichment or []),
        }
    answer = (
        f"{top_record['name']} is the guest associated with the Analytical Engine; "
        "her synthetic record lists mathematics and early computing as conversation topics."
        if top_record["id"] == "guest-ada"
        else f"{top_record['name']} is the best match in the synthetic guest corpus."
    )
    return {
        "answer": answer,
        "sources": [top_record["id"]],
        # Enrichment is carried in a separate field and cannot become a corpus citation.
        "live_enrichment": list(live_enrichment or []),
    }

rag_answer = compose_grounded_answer("analytical engine early computing")

evaluation_cases = [
    ("analytical engine early computing", "guest-ada"),
    ("compiler naval computing", "guest-grace"),
    ("trajectory orbital spaceflight", "guest-katherine"),
]
evaluation = {"queries": len(evaluation_cases), "top_1_correct": 0, "citation_preserved": 0}
for evaluation_query, expected_id in evaluation_cases:
    retrieved = retrieve_guests(evaluation_query, limit=1)
    evaluation["top_1_correct"] += int(retrieved[0]["id"] == expected_id)
    composed = compose_grounded_answer(evaluation_query)
    evaluation["citation_preserved"] += int(composed["sources"] == [expected_id])

## Deterministic assertions

These assertions verify the important behavior without a network, model, optional package, or credential.

In [2]:
assert [record["id"] for record in guest_corpus] == [
    "guest-ada", "guest-grace", "guest-katherine"
]
assert tokenize("Ada's Analytical Engine") == ["ada", "s", "analytical", "engine"]
assert [item["id"] for item in retrieve_guests("analytical engine programmer", 2)] == [
    "guest-ada", "guest-grace"
]
assert rag_answer == {
    "answer": "Ada Lovelace is the guest associated with the Analytical Engine; her synthetic record lists mathematics and early computing as conversation topics.",
    "sources": ["guest-ada"],
    "live_enrichment": [],
}
assert evaluation == {"queries": 3, "top_1_correct": 3, "citation_preserved": 3}
separated = compose_grounded_answer(
    "analytical engine",
    live_enrichment=[{"text": "Unverified current fact", "provenance": "optional web"}],
)
assert separated["sources"] == ["guest-ada"]
assert separated["live_enrichment"][0]["provenance"] == "optional web"

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

BM25-style scoring rewards query terms that are uncommon in the small corpus while normalizing for record length. The query about the Analytical Engine ranks `guest-ada` first. Composition uses only that record for its factual sentence and returns its ID in `sources`; optional current information remains in `live_enrichment` with separate provenance. The three-case evaluation checks both rank and citation behavior.

### Exercise

Add a fourth synthetic guest with a new stable ID and write one query that should rank that record first. Predict how its vocabulary changes inverse-document-frequency values, then extend the evaluation case list and assertion.

### Troubleshooting

If the wrong guest ranks first, inspect `tokenize`, each record's combined text, and the per-term document frequency. If an answer has no citation, verify that composition copies the retrieved record's `id` rather than a name guessed from the question. Treat `live_enrichment` as optional context, never as evidence for a `sources` entry.

## Credentials

Offline cells need no credential. Optional dataset loading, the CodeAgent, and Hub statistics check `ENABLE_LIVE_NETWORK` and `HF_TOKEN` before importing their packages. Optional web enrichment uses DuckDuckGo and checks only `ENABLE_LIVE_NETWORK` before importing the toolkit. No search API key is needed. Credential values are read from the environment and never printed.

In [3]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
hf_token = os.getenv("HF_TOKEN")
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 to load the optional Hub dataset.")
elif not hf_token:
    print("SKIPPED: Set HF_TOKEN to load the optional Hub dataset.")
else:
    try:
        from datasets import load_dataset
    except ImportError:
        print("SKIPPED: datasets is not installed; run `python -m pip install datasets`.")
    else:
        live_guest_dataset = load_dataset(
            "agents-course/unit3-invitees", split="train", token=hf_token
        )
        print(f"Loaded {len(live_guest_dataset)} optional live guest records.")

README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.32k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3 [00:00<?, ? examples/s]

Loaded 3 optional live guest records.


In [4]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 to run the optional CodeAgent.")
elif live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run the optional CodeAgent.")
else:
    try:
        from smolagents import CodeAgent, tool
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        @tool
        def live_guest_retriever(query: str) -> str:
            """Retrieve synthetic guest facts with stable citations.

            Args:
                query: Terms describing the guest to retrieve.
            """
            matches = guest_info_retriever(query)
            return "\n".join(
                f"[{match['id']}] {match['name']}: {match['description']}"
                for match in matches
            )

        if live_model_provider == "local":
            live_model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            live_model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token)
        live_alfred = CodeAgent(tools=[live_guest_retriever], model=live_model, max_steps=3)
        live_agent_answer = live_alfred.run(
            "Use the guest retriever to identify the Analytical Engine guest and cite the record ID."
        )
        print(live_agent_answer)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use the guest retriever to identify the Analytical Engine guest and cite the record ID.                         │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  guest_facts = live_guest_retriever(query="Analytical Engine")                                                    
  print(guest_facts)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[guest-ada] Ada Lovelace: Worked on Charles Babbage's Analytical Engine and is associated with early computer 
programming.
[guest-grace] Grace Hopper: Programming pioneer known for compiler work and clear explanations of computer systems.
[guest-katherine] Katherine Johnson: Mathematician whose trajectory calculations supported early crewed 
spaceflight.

Out: None

... (9 outputs left out)

[Step 3: Duration 112.30 seconds| Input tokens: 6,983 | Output tokens: 696]

Reached max steps.

[Step 4: Duration 34.63 seconds| Input tokens: 7,923 | Output tokens: 759]

The Analytical Engine was designed and built by Charles Babbage. It was a mechanical general-purpose computer that could perform complex mathematical operations and calculations. The Analytical Engine was developed between 1840 and 1864 and was considered one of the first computers to be built using mechanical components.


In [5]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
hf_token = os.getenv("HF_TOKEN")
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 to fetch optional Hub statistics.")
elif not hf_token:
    print("SKIPPED: Set HF_TOKEN to fetch optional Hub statistics.")
else:
    try:
        from huggingface_hub import HfApi
    except ImportError:
        print("SKIPPED: huggingface_hub is not installed; run `python -m pip install huggingface_hub`.")
    else:
        live_dataset_info = HfApi(token=hf_token).dataset_info(
            "agents-course/unit3-invitees"
        )
        live_hub_stats = {
            "dataset_id": live_dataset_info.id,
            "downloads": live_dataset_info.downloads,
            "likes": live_dataset_info.likes,
            "provenance": "optional live Hub metadata; not a retrieved guest fact",
        }
        print(live_hub_stats)

{'dataset_id': 'agents-course/unit3-invitees', 'downloads': 594, 'likes': 19, 'provenance': 'optional live Hub metadata; not a retrieved guest fact'}


In [6]:
import os

enable_live_network = os.getenv("ENABLE_LIVE_NETWORK") == "1"
if not enable_live_network:
    print("SKIPPED: Set ENABLE_LIVE_NETWORK=1 to run optional web enrichment.")
else:
    try:
        from smolagents import WebSearchTool
    except ImportError:
        print("SKIPPED: smolagents toolkit search is not installed; run `python -m pip install 'smolagents[toolkit]'`.")
    else:
        web_search = WebSearchTool(engine="duckduckgo")
        live_web_text = web_search("Ada Lovelace current public exhibition")
        live_web_enrichment = {
            "text": str(live_web_text),
            "provenance": "optional live web; not retrieved from the guest corpus",
        }
        print(live_web_enrichment)

{'text': '## Search Results\n\n[Ada Lovelace - Wikipedia](https://en.wikipedia.org/wiki/Ada_Lovelace)\n Ada  Lovelace ... Augusta Ada King, Countess of Lovelace (née Byron; 10 December 1815 - 27 November 1852), also known as Ada  Lovelace , was an English mathematician and writer chiefly known for work on Charles Babbage \'s proposed mechanical general-purpose computer, the analytical engine.\n\n[Ada Lovelace Exhibition: Pioneer of Computer Programming](https://musemap.art/en/madrid/whats-on/exhibitions/espacio-fundacion-telefonica-ada-lovelace-la-encantadora-de-numeros)\nDiscover the remarkable life and contributions of Ada  Lovelace , the visionary mathematician and first computer programmer. This exhibition celebrates her enduring legacy.\n\n[Ada Lovelace Tunnel Reopens at De Vere Horsley Estate in Surrey](https://theartsshelf.com/2026/05/02/ada-lovelace-tunnel-de-vere-horsley-estate-surrey/)\nA hidden 19th century tunnel linked to computing pioneer Ada  Lovelace has reopened to the

## Completion checklist

- [ ] Inspected stable guest IDs and tokenization.
- [ ] Ran retrieval, composition, and the three-case evaluation.
- [ ] Verified citations and optional enrichment stay separate.
- [ ] Reviewed every live guard before enabling network access.

## Attribution

The synthetic corpus, retriever, composer, and evaluation are original. The project structure and live dataset identifier follow the official [Hugging Face Agents Course Unit 3 guest-retrieval lesson](https://huggingface.co/learn/agents-course/en/unit3/agentic-rag/invitees). Current optional agent and search shapes follow the official [smolagents guided tour](https://huggingface.co/docs/smolagents/main/guided_tour), and Hub statistics follow the official [huggingface_hub API](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api). No upstream notebook or dataset records are reproduced. Notebook metadata carries the Apache-2.0 license.